# <font color="ffc800"> **[Piper](https://github.com/rhasspy/piper) model exporter.**
## ![Piper logo](https://contribute.rhasspy.org/img/logo.png)
---

* Notebook created by: [rmcpantoja](http://github.com/rmcpantoja)
* Collaborator: [Xx_Nessu_xX](http://github.com/XxNessuxX)
* Some changes made by [KiON-GiON](https://github.com/KiON-GiON)

In [ ]:
#@markdown # <font color="ffc800"> **Install software.** 📦
#@markdown ---

print("\033[93mInstalling...")
%cd /content
!git clone -q -b 2 https://github.com/KiON-GiON/piper1-gpl
%cd /content/piper1-gpl
!pip install -q -e ".[train]"
!pip install -q --upgrade gdown onnx onnxscript
print("\033[93mDone!")

In [ ]:
#@markdown # <font color="ffc800"> **Voice package generation section.** 🗣️
#@markdown ---
%cd /content/piper1-gpl

import json
import os
import re
import shutil
import subprocess
import tarfile
from collections import OrderedDict
from pathlib import Path

import gdown
import ipywidgets as widgets
from google.colab import output
from IPython.display import display

guideurl = "https://github.com/rmcpantoja/piper/blob/master/notebooks/wav/en"

#@markdown #### *Download:*
#@markdown **Drive ID or direct download link of the checkpoint (.ckpt):**
model_id = "" #@param {type:"string"}
#@markdown **Drive ID or direct download link of the config.json file:**
config_id = "" #@param {type:"string"}
#@markdown ---

#@markdown #### *Creation process:*
#@markdown **Choose or type the language code (e.g. es_ES, es_MX, en_US, ja_JP):**
language = "es_MX" #@param ["ar_JO", "ca_ES", "cs_CZ", "da_DK", "de_DE", "el_GR", "en_GB", "en_US", "es_ES", "es_MX", "es_LA", "fi_FI", "fr_FR", "grc", "hu_GU", "is_IS", "it_IT", "ja_JP", "kk_KZ", "ka_GE", "lb_LU", "nb", "ne", "nl_BE", "no_NO", "pl_PL", "pt_BR", "pt_PT", "ro_RO", "ru_RU", "sk_SK", "sr", "sv_SE", "sw_CD", "tr_TR", "uk_UA", "vi_VN", "zh_CN"] {allow-input: true}
voice_name = "" #@param {type:"string"}
quality = "medium" #@param ["high", "low", "medium", "x-low"]
#@markdown **Do you want to write a model card?** *(Optional.)*
write_model_card = False #@param {type:"boolean"}

#@markdown **Export streaming (+RT) encoder/decoder variant for Dengjen?**
streaming = False #@param {type:"boolean"}

#@markdown #### *Optional inference defaults override (leave empty to keep config.json values):*
noise_scale_override = "" #@param {type:"string"}
length_scale_override = "" #@param {type:"string"}
noise_w_override = "" #@param {type:"string"}

clean_voice_name = re.sub(r"[^a-z0-9_]+", "_", voice_name.strip().lower()).strip("_")
if not clean_voice_name:
    raise ValueError("Please enter a valid voice_name (letters/numbers/underscores).")

export_voice_name = (
    f"{language}-{clean_voice_name}+RT-{quality}"
    if streaming
    else f"{language}-{clean_voice_name}-{quality}"
)

project_dir = Path("/content/project")
export_voice_path = project_dir / f"voice-{export_voice_name}"
packages_path = project_dir / "packages"

# Limpiar exportaciones previas del mismo nombre para no mezclar .onnx antiguos
if export_voice_path.exists():
    shutil.rmtree(export_voice_path)
export_voice_path.mkdir(parents=True, exist_ok=True)
packages_path.mkdir(parents=True, exist_ok=True)

ckpt_path = project_dir / "model.ckpt"
json_path = export_voice_path / f"{export_voice_name}.onnx.json"


def download_asset(source: str, dest: Path) -> None:
    source = source.strip()
    if not source:
        raise ValueError(f"Empty download URL/ID for {dest.name}")
    if dest.exists():
        dest.unlink()
    is_drive_id = "/" not in source and len(source) > 15
    if "drive.google.com" in source or is_drive_id:
        if is_drive_id:
            gdown.download(id=source, output=str(dest), quiet=False)
        else:
            gdown.download(source, str(dest), quiet=False)
    else:
        subprocess.run(["wget", "-q", source, "-O", str(dest)], check=True)
    if not dest.exists() or dest.stat().st_size == 0:
        raise RuntimeError(f"Failed to download {dest.name}; verify permissions/link.")


def play_cue(name: str) -> None:
    try:
        output.eval_js(f'new Audio("{guideurl}/{name}.wav?raw=true").play()')
    except Exception:
        pass


def start_process(is_streaming: bool) -> None:
    if not ckpt_path.exists():
        raise FileNotFoundError("Missing /content/project/model.ckpt")
    play_cue("starting")

    if not is_streaming:
        subprocess.run(
            [
                "python", "-m", "piper.train.export_onnx",
                "--checkpoint", str(ckpt_path),
                "--output-file", str(export_voice_path / f"{export_voice_name}.onnx"),
            ],
            check=True,
        )
    else:
        subprocess.run(
            [
                "python", "-m", "piper.train.export_onnx_streaming",
                "--checkpoint", str(ckpt_path),
                "--output-dir", str(export_voice_path),
            ],
            check=True,
        )

    print("\033[93mCompressing...")
    tar_path = packages_path / f"{export_voice_name}.tar.gz"
    with tarfile.open(tar_path, "w:gz") as tar:
        for item in sorted(export_voice_path.iterdir()):
            if item.is_file():
                tar.add(item, arcname=item.name)

    play_cue("success")
    print(f"\033[93mDone! Package ready: {tar_path}")


print("\033[93mDownloading checkpoint and config...")
download_asset(model_id, ckpt_path)
download_asset(config_id, json_path)

with open(json_path, "r", encoding="utf-8") as f:
    config = json.load(f, object_pairs_hook=OrderedDict)

inference = config.setdefault("inference", OrderedDict())
if noise_scale_override.strip():
    inference["noise_scale"] = float(noise_scale_override)
if length_scale_override.strip():
    inference["length_scale"] = float(length_scale_override)
if noise_w_override.strip():
    inference["noise_w"] = float(noise_w_override)

if streaming:
    config["streaming"] = True
    config["key"] = export_voice_name
else:
    config.pop("streaming", None)

with open(json_path, "w", encoding="utf-8") as f_new:
    json.dump(config, f_new, indent=4, ensure_ascii=False)

if write_model_card:
    sample_rate = config["audio"]["sample_rate"]
    num_speakers = config.get("num_speakers", 1)
    play_cue("waiting")
    text_area = widgets.Textarea(
        description="Model card:",
        value=(
            f"# Model card for {clean_voice_name} ({quality})\n\n"
            f"* Language: {language}\n"
            f"* Speakers: {num_speakers}\n"
            f"* Quality: {quality}\n"
            f"* Samplerate: {sample_rate}Hz\n\n"
            "## Dataset\n\n* URL: \n* License: \n\n"
            "## Training\n\nTrained from scratch.\nOr finetuned from: "
        ),
        layout=widgets.Layout(width="500px", height="200px"),
    )
    button = widgets.Button(description="Start")

    def create_model_card(_button):
        model_card_text = text_area.value.strip()
        (export_voice_path / "MODEL_CARD").write_text(model_card_text, encoding="utf-8")
        text_area.close()
        button.close()
        output.clear()
        start_process(streaming)

    button.on_click(create_model_card)
    display(text_area, button)
else:
    start_process(streaming)

In [ ]:
#@markdown # <font color="ffc800"> **Download/export your generated voice package.** 📥
#@markdown ---

import shutil
from pathlib import Path

#@markdown #### *How do you want to export your model?*
export_mode = "Download the voice package on my device (may take some time)" #@param ["Download the voice package on my device (may take some time)", "upload it to my Google Drive"]
tar_file = packages_path / f"{export_voice_name}.tar.gz"

print("\033[93mExporting package...")
if export_mode == "Download the voice package on my device (may take some time)":
    from google.colab import files
    files.download(str(tar_file))
    msg = "Please wait a moment while the package is being downloaded."
else:
    from google.colab import drive
    drive.mount("/content/drive")
    voicepacks_folder = Path("/content/drive/MyDrive/piper voice packages")
    voicepacks_folder.mkdir(parents=True, exist_ok=True)
    shutil.copy2(tar_file, voicepacks_folder / tar_file.name)
    msg = f"You can find the generated voice package at: {voicepacks_folder / tar_file.name}."
print(f"\033[93mDone! {msg}")

# "*I want to test this model! I don't need anything else anymore?*"

No, this is almost the end! Now you can share your generated package to your friends, upload to a cloud storage and/or test it on:
* [The inference notebook](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_inference_(ONNX).ipynb) (**Outdated**)
  * run the cells in order for it to work correctly, as well as all the notebooks. Also, the inference notebook will guide you through the process using the enhanced accessibility feature if you wish. It's easy to use. Test it!
* Or through the NVDA screen reader!
  * Download and install the latest version of the [add-on](https://github.com/mush42/sonata-nvda/releases).
  * Once the add-on is installed, go to NVDA menu/Sonata voice manager...
  * In the installed voices page, tab until you find the `Install from local file` button, press enter and select the generated package in your downloads.
  * Once the package is selected and installed, apply the changes and restart NVDA to update the voice list.
* Enjoy your creation!